In [1]:
import ollama

In [2]:
inventory_db = {
        "laptop": {"stock": 5, "base_price": 1200},
        "monitor": {"stock": 0, "base_price": 300},
        "keyboard": {"stock": 25, "base_price": 80}
    }

In [3]:
def check_inventory(product_name):
    """Return stock and base_price for a product from inventory_db."""
    product_name = product_name.lower()
    if product_name in inventory_db:
        return inventory_db[product_name]
    return {"stock": 0, "base_price": None}

In [4]:
def calculate_loyalty_discount(base_price, years_as_customer):
    """Calculate final price applying loyalty discount.
    Ensures base_price is treated as a number even if passed as a string.
    """
    # Convert base_price to float in case it's a string
    try:
        base_price = float(base_price)
    except (TypeError, ValueError):
        # If conversion fails, raise a clear error
        raise ValueError(f"base_price must be a number, got {base_price!r}")

    discount = min(years_as_customer * 0.05, 0.30)  # max 30%
    final_price = base_price * (1 - discount)
    return round(final_price, 2)

In [5]:
# 1. Map the functions so the script can call them by name
available_functions = {
    'check_inventory': check_inventory,
    'calculate_loyalty_discount': calculate_loyalty_discount,
}

In [6]:
#DEFINING SCHEMA
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Get stock and price for a product",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"}
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_loyalty_discount",
            "description": "Calculate final price based on loyalty years",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "number"},
                    "years_as_customer": {"type": "integer"}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [7]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number'},
     'years_as_customer': {'type': 'integer'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [8]:
#“I am a customer for 5 years. What will be the final price of a laptop?"
#"I want to buy an iPhone. Can you check stock?"
message=[
        {"role": "user", "content":"I am a customer for 5 years. What will be the final price of a laptop?"}
    ]

#CALLING llM
response = ollama.chat(
    model="llama3.2:latest",
    messages=message
   ,
    tools=tools
)

print(response)

model='llama3.2:latest' created_at='2026-04-13T07:35:45.7953868Z' done=True done_reason='stop' total_duration=4837914500 load_duration=281751500 prompt_eval_count=215 prompt_eval_duration=236724500 eval_count=30 eval_duration=4243700000 message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='calculate_loyalty_discount', arguments={'base_price': '1000', 'years_as_customer': '5'}))]) logprobs=None


In [9]:
response["message"]

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='calculate_loyalty_discount', arguments={'base_price': '1000', 'years_as_customer': '5'}))])

In [10]:
# Check if the model asked to use any tools
tool_calls = response['message'].get('tool_calls')

if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call['function']['name']
        tool_args = tool_call['function']['arguments']

        # Get the actual Python function
        function_to_call = available_functions[tool_name]

        # Convert arguments to proper types
        # For base_price: ensure it's a number
        if 'base_price' in tool_args:
            tool_args['base_price'] = float(tool_args['base_price'])
        # For years_as_customer: ensure it's an integer
        if 'years_as_customer' in tool_args:
            tool_args['years_as_customer'] = int(tool_args['years_as_customer'])

        # Run the function with given arguments
        result = function_to_call(**tool_args)
        print(f"Tool: {tool_name}, Args: {tool_args}, Result: {result}")

        # Add the model's tool request to the conversation
        message.append(response['message'])

        # Add the tool's result back to the conversation
        message.append({
            'role': 'tool',
            'content': str(result)
        })

message

Tool: calculate_loyalty_discount, Args: {'base_price': 1000.0, 'years_as_customer': 5}, Result: 750.0


[{'role': 'user',
  'content': 'I am a customer for 5 years. What will be the final price of a laptop?'},
 Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='calculate_loyalty_discount', arguments={'base_price': 1000.0, 'years_as_customer': 5}))]),
 {'role': 'tool', 'content': '750.0'}]

In [11]:
# Ask the model to generate the final answer
final_response = ollama.chat(
    model='llama3.2:latest',
    messages=message
)

print(final_response['message']['content'])

The final price of the laptop would be $750.00, which is a discount of 25% from the original base price of $1,000. This represents a loyalty discount for being a customer for 5 years.
